Imports

In [ ]:
import re
import pickle
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from rake_nltk import Rake

# ----------------------------
# Reproducibility (optional)
# ----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

Load CSVs + embeddings

In [ ]:
# ----------------------------
# 1) LOAD DATASETS
# ----------------------------
main_df  = pd.read_csv(r'/teamspace/studios/this_studio/datasets/main_mixed_df.csv')
dialog_df= pd.read_csv(r'/teamspace/studios/this_studio/datasets/dialogue.csv')
docs_df  = pd.read_csv(r'/teamspace/studios/this_studio/datasets/documents.csv')
resp_df  = pd.read_csv(r'/teamspace/studios/this_studio/datasets/response.csv')
summ_df  = pd.read_csv(r'/teamspace/studios/this_studio/datasets/summaries.csv')

# ----------------------------
# 2) LOAD EMBEDDINGS (id -> np.array)
# ----------------------------
def load_embeddings(file_path):
    with open(file_path, 'rb') as f:
        return pickle.load(f)

emb_dialog = load_embeddings(r'/teamspace/studios/this_studio/embeddings/dialog_embeddings.pkl')
emb_docs   = load_embeddings(r'/teamspace/studios/this_studio/embeddings/doc_embeddings.pkl')
emb_resp   = load_embeddings(r'/teamspace/studios/this_studio/embeddings/response_embeddings.pkl')
emb_summ   = load_embeddings(r'/teamspace/studios/this_studio/embeddings/summ_embeddings.pkl')

print("Loaded main_df columns:")
print(list(main_df.columns))
print("Example row:")
print(main_df.iloc[0])

Core helpers: parse sequences, canonical IDs, null handling

In [ ]:
# Matches canonical ids like: dl-12, d-7, r-3, s-10
CANON_RE = re.compile(r"^(dl|d|r|s)-\d+$", re.IGNORECASE)

def parse_serialized_sequence(s):
    """
    Parse serialized list strings like:
      '["dl-1","d-2","r-1"]'   OR  '[dl-1,d-2,r-1]'
    into Python list of strings.

    Your data looks like:
      "[dl-1,d-2,r-1,...]"
    so we keep it simple and robust.
    """
    if not isinstance(s, str):
        return []
    s = s.strip()
    if not (s.startswith("[") and s.endswith("]")):
        return []
    items = [x.strip().strip('"').strip("'") for x in s[1:-1].split(",")]
    return [x for x in items if x]

def is_nullish(x):
    """
    True if x is None/NaN/'NULL'/'None'/'' etc.
    """
    if x is None:
        return True
    if isinstance(x, float) and np.isnan(x):
        return True
    s = str(x).strip().lower()
    return s in {"", "null", "none", "nan"}

def normalize_id(x):
    """
    Standardize id strings (strip spaces).
    """
    return str(x).strip()

def to_canonical(prefix: str, raw_id):
    """
    Prevents double-prefixing:
    - If raw_id already canonical like 'd-1', keep it
    - Else convert to '<prefix>-<raw_id>'
    """
    s = str(raw_id).strip()
    if CANON_RE.match(s):
        return s
    return f"{prefix}-{s}"

Build vocabularies: users, actions, nodes, tasks

In [ ]:
# ----------------------------
# 1) USER VOCAB
# ----------------------------
unique_users = sorted(main_df["userid"].dropna().astype(str).unique())
user2idx = {u: i for i, u in enumerate(unique_users)}
idx2user = {i: u for u, i in user2idx.items()}

# ----------------------------
# 2) ACTION VOCAB (from interactions)
# ----------------------------
all_actions = set()
for seq in main_df["interactions"].dropna():
    all_actions.update(parse_serialized_sequence(seq))

actions2idx = {a: i for i, a in enumerate(sorted(all_actions))}
idx2action  = {i: a for a, i in actions2idx.items()}

# ----------------------------
# 3) NODE VOCAB
#   - from text_nodes sequences
#   - plus query_text_id and target_output_id
# ----------------------------
node_ids = set()

for seq in main_df["text_nodes"].dropna():
    for nid in parse_serialized_sequence(seq):
        if not is_nullish(nid):
            node_ids.add(normalize_id(nid))

for col in ["query_text_id", "target_output_id"]:
    if col in main_df.columns:
        for x in main_df[col].dropna():
            if not is_nullish(x):
                node_ids.add(normalize_id(x))

node2idx = {nid: i for i, nid in enumerate(sorted(node_ids))}
idx2node = {i: nid for nid, i in node2idx.items()}

# ----------------------------
# 4) TASK VOCAB + task_id
# ----------------------------
assert "task" in main_df.columns, "main_df must contain 'task'"
task_names = sorted(main_df["task"].dropna().astype(str).unique())
task2idx = {t: i for i, t in enumerate(task_names)}
idx2task = {i: t for t, i in task2idx.items()}

main_df["task_id"] = main_df["task"].astype(str).map(task2idx)
assert main_df["task_id"].isna().sum() == 0, "Unmapped task found"

n_tasks = len(task2idx)

print("VOCAB SIZES:")
print("  users:", len(user2idx))
print("  actions:", len(actions2idx), "->", sorted(actions2idx.keys()))
print("  nodes:", len(node2idx))
print("  tasks:", n_tasks, "->", task2idx)

Build id2text using canonical IDs (no double-prefix)

In [ ]:
def build_id2text(dialog_df, docs_df, resp_df, summ_df):
    """
    Create mapping canonical_id -> text for ALL text-bearing tables.
    Keys will be like:
      dl-<id>, d-<id>, r-<id>, s-<id>
    """
    id2text = {}

    # Dialogue
    if "dialog_id" in dialog_df.columns:
        for _, r in dialog_df.dropna(subset=["dialog_id"]).iterrows():
            cid = to_canonical("dl", r["dialog_id"])
            id2text[cid] = str(r.get("dialogue_text", ""))

    # Docs
    if "NewsID" in docs_df.columns:
        for _, r in docs_df.dropna(subset=["NewsID"]).iterrows():
            cid = to_canonical("d", r["NewsID"])
            nb = str(r.get("NewsBody", "")).strip()
            hl = str(r.get("Headline", "")).strip()
            id2text[cid] = nb if nb else hl

    # Responses
    if "response_id" in resp_df.columns:
        for _, r in resp_df.dropna(subset=["response_id"]).iterrows():
            cid = to_canonical("r", r["response_id"])
            id2text[cid] = str(r.get("response_text", ""))

    # Summaries
    if "SummID" in summ_df.columns:
        for _, r in summ_df.dropna(subset=["SummID"]).iterrows():
            cid = to_canonical("s", r["SummID"])
            id2text[cid] = str(r.get("Summary", ""))

    return id2text

id2text = build_id2text(dialog_df, docs_df, resp_df, summ_df)

print("id2text size:", len(id2text))
print("sample keys:", list(id2text.keys())[:10])
id2text

Offline id2phrases + keyphrase vocabulary kp2idx

In [ ]:
from tqdm.auto import tqdm
from rake_nltk import Rake

def build_id2phrases_and_vocab(id2text, min_words=1):
    """
    For each canonical id:
      - run RAKE once
      - store id2phrases[id] = list of phrases (deduped)
    Then build kp2idx as union of all phrases.
    """
    rake = Rake()
    id2phrases = {}
    vocab = set()

    # tqdm progress bar over ids
    for cid, text in tqdm(id2text.items(),
                          desc="Extracting RAKE keyphrases",
                          total=len(id2text)):

        t = str(text).strip()
        if not t:
            id2phrases[cid] = []
            continue

        rake.extract_keywords_from_text(t)
        phrases = rake.get_ranked_phrases()
        phrases = [p for p in phrases if len(p.split()) >= min_words]

        # dedupe preserve order
        seen = set()
        uniq = []
        for p in phrases:
            if p not in seen:
                seen.add(p)
                uniq.append(p)

        id2phrases[cid] = uniq
        vocab.update(uniq)

    # tqdm for vocab building (optional but useful if huge vocab)
    sorted_vocab = sorted(vocab)
    kp2idx = {p: i for i, p in enumerate(sorted_vocab)}
    idx2kp = {i: p for p, i in kp2idx.items()}

    return id2phrases, kp2idx, idx2kp


id2phrases, kp2idx, idx2kp = build_id2phrases_and_vocab(id2text, min_words=1)

print("Keyphrase vocab size:", len(kp2idx))
id2phrases


In [ ]:
from collections import Counter
import numpy as np

def summarize_keyphrases(id2phrases):
    counts = {cid: len(phrases) for cid, phrases in id2phrases.items()}
    
    values = list(counts.values())
    
    summary = {
        "num_ids": len(counts),
        "total_keyphrases": sum(values),
        "min_per_id": min(values) if values else 0,
        "max_per_id": max(values) if values else 0,
        "mean_per_id": float(np.mean(values)) if values else 0.0,
        "median_per_id": float(np.median(values)) if values else 0.0,
        "distribution": Counter(values)  # e.g., how many IDs have 0,1,2,... phrases
    }
    
    return counts, summary


counts_per_id, summary = summarize_keyphrases(id2phrases)

print("Keyphrases per ID:", counts_per_id)
print("\nSummary:")
for k, v in summary.items():
    print(k, ":", v)


In [ ]:
kp2idx

KP supervision rule: use target_output_id for generation tasks

In [ ]:
GEN_TASKS = {"dialogue_generation", "personalized_summarization"}

def get_kp_supervision_id(row):
    """
    For generation tasks:
      - use target_output_id (r-* for dialogue, s-* for summary)
    Else:
      - no KP supervision (return None)
    """
    task = str(row["task"]).strip()
    if task in GEN_TASKS:
        tid = row.get("target_output_id", None)
        if not is_nullish(tid):
            return normalize_id(tid)
    return None

def build_kp_target_from_id(kp_id, id2phrases, kp2idx, device):
    """
    kp_id: canonical id like 'r-2' or 's-10'
    returns (multi_hot_vector, gold_phrase_list)
    """
    phrases = id2phrases.get(kp_id, [])
    y = torch.zeros(len(kp2idx), device=device)
    for p in phrases:
        j = kp2idx.get(p, None)
        if j is not None:
            y[j] = 1.0
    return y, phrases

Embedding resolver for canonical ids

In [ ]:
class EmbLookup:
    @staticmethod
    def lookup(emb_dict, key):
        if key in emb_dict:
            return emb_dict[key]
        s = str(key)
        if s in emb_dict:
            return emb_dict[s]
        # numeric tail fallback
        nums = re.findall(r"\d+", s)
        if nums:
            n = nums[-1]
            if n in emb_dict:
                return emb_dict[n]
            try:
                ni = int(n)
                if ni in emb_dict:
                    return emb_dict[ni]
            except Exception:
                pass
        return None

class CanonicalEmbeddingResolver:
    """
    Input: canonical id string (dl-7, d-3, r-2, s-10)
    Output: np.array shape (768,)
    """
    def __init__(self, emb_dialog, emb_docs, emb_resp, emb_summ, d=768):
        self.emb_dialog = emb_dialog
        self.emb_docs = emb_docs
        self.emb_resp = emb_resp
        self.emb_summ = emb_summ
        self.d = d

    def get(self, cid: str) -> np.ndarray:
        cid = normalize_id(cid)
        prefix = cid.split("-")[0].lower()

        if prefix == "dl":
            v = EmbLookup.lookup(self.emb_dialog, cid)
        elif prefix == "d":
            v = EmbLookup.lookup(self.emb_docs, cid)
        elif prefix == "r":
            v = EmbLookup.lookup(self.emb_resp, cid)
        elif prefix == "s":
            v = EmbLookup.lookup(self.emb_summ, cid)
        else:
            v = None

        if v is None:
            return np.zeros((self.d,), dtype=np.float32)

        v = np.asarray(v, dtype=np.float32)
        if v.shape[-1] != self.d:
            raise ValueError(f"Embedding dim mismatch for {cid}: got {v.shape}, expected ({self.d},)")
        return v

emb_resolver = CanonicalEmbeddingResolver(emb_dialog, emb_docs, emb_resp, emb_summ, d=768)

UniHis SSM Encoder

In [ ]:
class UniHisSSMEncoder(nn.Module):
    def __init__(
        self,
        vocab_sizes,
        d_model=768,
        d_state=64,
        n_tasks=8,
        n_users=10000,
        d_qtext=768,
        known_task=True
    ):
        super().__init__()
        self.d_state = d_state
        self.known_task = known_task

        def init_eig():
            Q = nn.Parameter(torch.randn(d_state, d_state) * 0.02)
            Lambda = nn.Parameter(torch.empty(d_state).uniform_(-1.0, -0.1))
            return Q, Lambda

        self.Q_L, self.Lambda_L = init_eig()
        self.Q_S, self.Lambda_S = init_eig()
        self.Q_E, self.Lambda_E = init_eig()

        self.B = nn.Parameter(torch.randn(d_state, d_model))
        self.tau_S = nn.Parameter(torch.tensor(1.0))
        self.tau_E = nn.Parameter(torch.tensor(1.0))

        # ========= ACTION → LATENT MOOD ==========
        self.action_proj = nn.Linear(vocab_sizes["actions"], d_model)

        self.MLP_mood = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, d_model)
        )

        self.W_a = nn.Linear(d_model, d_state)

        self.Mixer_L = nn.Linear(d_state, d_state)
        self.Mixer_S = nn.Linear(d_state, d_state)
        self.Mixer_E = nn.Linear(d_state, d_state)

        self.W_fuse = nn.Linear(3 * d_state, 3)
        self.Mixer_Fuse = nn.Linear(d_state, d_state)

        self.action_head = nn.Linear(3 * d_state, vocab_sizes["actions"])
        self.node_head   = nn.Linear(3 * d_state, vocab_sizes["nodes"])

        self.Embed_Q = nn.Linear(d_qtext, d_state)
        self.W_task  = nn.Linear(2 * d_state, n_tasks)
        self.log_var_task = nn.Parameter(torch.zeros(()))

        self.W_hyp = nn.Linear(2 * d_state, d_state)
        self.W_theta = nn.ModuleList([
            nn.Linear(d_state, 2 * d_state) for _ in range(n_tasks)
        ])

        self.W_user = nn.Linear(d_state, n_users)
        self.criterion = nn.CrossEntropyLoss()

    def make_A(self, Q, Lambda_diag):
        Q_inv = torch.linalg.inv(Q)
        return Q @ torch.diag(Lambda_diag) @ Q_inv

    def forward(
        self,
        e_seq,
        action_onehot=None,     # <-- ADD THIS
        action_labels=None,
        node_labels=None,
        task_labels=None,
        user_labels=None,
        q_text_emb=None,
        kp_targets=None,
        kp_loss_mask=None
    ):
        B, T, _ = e_seq.shape
        device = e_seq.device

        xL = torch.zeros(B, self.d_state, device=device)
        xS = torch.zeros(B, self.d_state, device=device)
        xE = torch.zeros(B, self.d_state, device=device)

        act_logits, node_logits = [], []

        A_L = self.make_A(self.Q_L, torch.exp(self.Lambda_L))

        eps_S = torch.sigmoid(self.tau_S)
        k = torch.log((1 - eps_S) / eps_S) / torch.abs(self.Lambda_S)
        A_S = self.make_A(self.Q_S, torch.sigmoid(-k * self.Lambda_S))

        eta = (torch.exp(-1.0 / self.tau_E) - 1.0) / self.Lambda_E
        A_E = self.make_A(self.Q_E, 1.0 + eta * self.Lambda_E)

        for t in range(T):
            e_t = e_seq[:, t]

            # mood = torch.sigmoid(self.W_a(self.MLP_mood(e_t)))
            # u_t = torch.bmm(
            #     (self.B.unsqueeze(0) * mood.unsqueeze(2)),
            #     e_t.unsqueeze(2)
            # ).squeeze(2)

            # =========================
            # ACTION → ONE-HOT → EMBEDDING
            # =========================
            if action_onehot is None:
                raise ValueError("action_onehot is required for mood modulation")

            a_t = action_onehot[:, t, :]                   # (B, |A|)
            a_emb = self.action_proj(a_t)                  # (B, d_model)

            # =========================
            # MLP mood from ACTION
            # =========================
            mood_vec = self.MLP_mood(a_emb)                # (B, d_model)

            # =========================
            # PROJECT mood into STATE SPACE LIKE TEXT
            # =========================
            mood = torch.sigmoid(self.W_a(mood_vec))       # (B, d_state)

            # =========================
            # MODULATE MATRIX INPUT
            # =========================
            u_t = torch.bmm(
                (self.B.unsqueeze(0) * mood.unsqueeze(2)),
                e_t.unsqueeze(2)
            ).squeeze(2)


            xL = torch.matmul(xL, A_L.T) + u_t
            xS = torch.matmul(xS, A_S.T) + u_t
            xE = torch.matmul(xE, A_E.T) + u_t

            x_cat = torch.cat([xL, xS, xE], dim=-1)
            act_logits.append(self.action_head(x_cat))
            node_logits.append(self.node_head(x_cat))

        act_logits  = torch.stack(act_logits, dim=1)
        node_logits = torch.stack(node_logits, dim=1)

        yL = self.Mixer_L(xL)
        yS = self.Mixer_S(xS)
        yE = self.Mixer_E(xE)

        w = F.softmax(self.W_fuse(torch.cat([xL, xS, xE], dim=-1)), dim=-1)
        h_hist = self.Mixer_Fuse(
            w[:, 0:1] * yL +
            w[:, 1:2] * yS +
            w[:, 2:3] * yE
        )

        out = {
            "action_logits": act_logits,
            "node_logits": node_logits,
            "pred_actions": act_logits.argmax(dim=-1),
            "pred_nodes": node_logits.argmax(dim=-1),
            "h_hist": h_hist
        }

        total_loss = torch.tensor(0.0, device=device)

        if action_labels is not None and node_labels is not None:
            loss_action = self.criterion(
                act_logits.view(-1, act_logits.size(-1)),
                action_labels.view(-1)
            )
            loss_node = self.criterion(
                node_logits.view(-1, node_logits.size(-1)),
                node_labels.view(-1)
            )
            total_loss = total_loss + loss_action + loss_node
            out.update({"loss_action": loss_action, "loss_node": loss_node})

        if q_text_emb is not None and task_labels is not None:
            q_emb = self.Embed_Q(q_text_emb)
            z_switch = torch.cat([h_hist, q_emb], dim=-1)

            task_logits = self.W_task(z_switch)
            P_task = F.softmax(task_logits, dim=-1)

            ce_task = self.criterion(task_logits, task_labels)
            loss_task = torch.exp(-self.log_var_task) * ce_task + self.log_var_task
            total_loss = total_loss + loss_task

            out.update({
                "task_logits": task_logits,
                "P_task": P_task,
                "pred_task": P_task.argmax(dim=-1),
                "true_task": task_labels,
                "loss_task": loss_task
            })

            h_hyp = F.relu(self.W_hyp(z_switch))

            if self.known_task:
                theta = self.W_theta[task_labels[0]](h_hyp)
                g, b = theta.chunk(2, dim=-1)
                h_final = torch.sigmoid(g) * h_hist + b
            else:
                h_final = 0.0
                for i in range(len(self.W_theta)):
                    theta_i = self.W_theta[i](h_hyp)
                    g_i, b_i = theta_i.chunk(2, dim=-1)
                    h_final = h_final + P_task[:, i:i+1] * (torch.sigmoid(g_i) * h_hist + b_i)
        else:
            h_final = h_hist

        if user_labels is not None:
            user_logits = self.W_user(h_final)
            P_user = F.softmax(user_logits, dim=-1)
            loss_user = self.criterion(user_logits, user_labels)
            total_loss = total_loss + loss_user

            out.update({
                "user_logits": user_logits,
                "P_user": P_user,
                "pred_user": P_user.argmax(dim=-1),
                "true_user": user_labels,
                "loss_user": loss_user
            })

        out["loss"] = total_loss
        out["h_final"] = h_final
        return out

RWKV keyphrase decoder + combined model

In [ ]:
class LensGate(nn.Module):
    def __init__(self, d_state, d_qtext):
        super().__init__()
        self.Wq = nn.Linear(d_qtext, d_state)
        self.Wfilm = nn.Linear(d_state, 2 * d_state)
        self.Wg = nn.Linear(2 * d_state, d_state)

    def forward(self, h_final, q_text_emb):
        q0 = self.Wq(q_text_emb)
        gamma, beta = self.Wfilm(h_final).chunk(2, dim=-1)
        q_lens = (1.0 + gamma) * q0 + beta
        g = torch.sigmoid(self.Wg(torch.cat([h_final, q_lens], dim=-1)))
        return g * q_lens + (1.0 - g) * q0

class RWKVTimeMix(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.time_mix = nn.Parameter(torch.rand(d))
        self.key = nn.Linear(d, d, bias=False)
        self.value = nn.Linear(d, d, bias=False)
        self.receptance = nn.Linear(d, d, bias=False)
        self.output = nn.Linear(d, d, bias=False)
        self.ln = nn.LayerNorm(d)

    def forward(self, x):
        x = self.ln(x)
        x_shift = torch.zeros_like(x)
        x_shift[:, 1:] = x[:, :-1]
        xm = x * self.time_mix + x_shift * (1 - self.time_mix)

        r = torch.sigmoid(self.receptance(xm))
        k = self.key(xm)
        v = self.value(xm)

        w = torch.softmax(k, dim=1)
        mix = torch.sum(w * v, dim=1, keepdim=True)
        y = self.output(r * mix).expand_as(x)
        return x + y

class RWKVChannelMix(nn.Module):
    def __init__(self, d, expansion=4):
        super().__init__()
        self.time_mix = nn.Parameter(torch.rand(d))
        self.key = nn.Linear(d, expansion*d, bias=False)
        self.value = nn.Linear(expansion*d, d, bias=False)
        self.receptance = nn.Linear(d, d, bias=False)
        self.ln = nn.LayerNorm(d)

    def forward(self, x):
        x = self.ln(x)
        x_shift = torch.zeros_like(x)
        x_shift[:, 1:] = x[:, :-1]
        xm = x * self.time_mix + x_shift * (1 - self.time_mix)

        r = torch.sigmoid(self.receptance(xm))
        k = torch.relu(self.key(xm))
        v = self.value(k)
        return x + r * v

class RWKVKeyphraseDecoder(nn.Module):
    def __init__(self, d_state, d_qtext, n_tasks, n_keyphrases, n_layers=2):
        super().__init__()
        self.d_state = d_state
        self.n_tasks = n_tasks

        self.lens_gate = LensGate(d_state, d_qtext)

        self.task_embed = nn.Embedding(n_tasks, d_state)
        self.task_proj = nn.Linear(d_state, d_state)

        self.proj_h = nn.Linear(d_state, d_state)
        self.proj_q = nn.Linear(d_state, d_state)

        self.blocks = nn.ModuleList([])
        for _ in range(n_layers):
            self.blocks.append(RWKVTimeMix(d_state))
            self.blocks.append(RWKVChannelMix(d_state))

        self.out_ln = nn.LayerNorm(d_state)
        self.kp_head = nn.Linear(d_state, n_keyphrases)
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, h_final, q_text_emb, P_task=None, kp_targets=None):
        B = h_final.size(0)

        q_cond = self.lens_gate(h_final, q_text_emb)

        if P_task is not None:
            task_ids = torch.arange(self.n_tasks, device=h_final.device)
            Temb = self.task_embed(task_ids)
            prompt = P_task @ Temb
            prompt = self.task_proj(prompt)
        else:
            prompt = torch.zeros(B, self.d_state, device=h_final.device)

        token_h = self.proj_h(h_final + prompt)
        token_q = self.proj_q(q_cond + prompt)
        x = torch.stack([token_h, token_q], dim=1)  # (B,2,d)

        for blk in self.blocks:
            x = blk(x)

        h_dec = self.out_ln(x[:, -1, :])
        kp_logits = self.kp_head(h_dec)

        out = {"kp_logits": kp_logits}
        if kp_targets is not None:
            out["loss_kp"] = self.bce(kp_logits, kp_targets)
        return out

    @torch.no_grad()
    def decode(self, kp_logits, idx2kp, topk=20, threshold=0.5):
        probs = torch.sigmoid(kp_logits)
        out = []
        for b in range(probs.size(0)):
            p = probs[b]
            vals, inds = torch.topk(p, k=min(topk, p.numel()))
            phrases = []
            for v, i in zip(vals.tolist(), inds.tolist()):
                if v >= threshold:
                    phrases.append((idx2kp[int(i)], float(v)))
            out.append(phrases)
        return out

class UniHisSSMWithKeyphrases(nn.Module):
    def __init__(self, encoder, d_qtext, n_tasks, n_keyphrases, rwkv_layers=2):
        super().__init__()
        self.encoder = encoder
        self.decoder = RWKVKeyphraseDecoder(
            d_state=encoder.d_state,
            d_qtext=d_qtext,
            n_tasks=n_tasks,
            n_keyphrases=n_keyphrases,
            n_layers=rwkv_layers
        )

    def forward(
    self,
    e_seq,
    action_onehot=None,
    action_labels=None,
    node_labels=None,
    task_labels=None,
    user_labels=None,
    q_text_emb=None,
    kp_targets=None,
    kp_loss_mask=None
):
        
        
        enc_out = self.encoder(
        e_seq=e_seq,
        action_onehot=action_onehot,   # <-- JUST PASS THROUGH
        action_labels=action_labels,
        node_labels=node_labels,
        task_labels=task_labels,
        user_labels=user_labels,
        q_text_emb=q_text_emb
        )

        h_final = enc_out["h_final"]
        P_task = enc_out.get("P_task", None)

        dec_out = self.decoder(
            h_final=h_final,
            q_text_emb=q_text_emb,
            P_task=P_task,
            kp_targets=kp_targets
        )

        out = {**enc_out, **dec_out}

        total = out["loss"]
        if "loss_kp" in out:
            if kp_loss_mask is None:
                total = total + out["loss_kp"]
            else:
                total = total + out["loss_kp"] * kp_loss_mask.mean()

        out["loss_total"] = total
        return out

Instantiate model correctly

In [ ]:
# Encoder head sizes
vocab_sizes = {
    "actions": len(actions2idx),
    "nodes": len(node2idx),
}

n_users_model = len(user2idx)  # IMPORTANT: avoid out-of-range user IDs

encoder = UniHisSSMEncoder(
    vocab_sizes=vocab_sizes,
    d_model=768,
    d_state=64,
    n_tasks=n_tasks,
    n_users=n_users_model,
    d_qtext=768,
    known_task=False
)

model = UniHisSSMWithKeyphrases(
    encoder=encoder,
    d_qtext=768,
    n_tasks=n_tasks,
    n_keyphrases=len(kp2idx),
    rwkv_layers=2
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
optim = torch.optim.AdamW(model.parameters(), lr=3e-4)

print("Model ready on:", device)


Sample 1-row forward pass (uses text_nodes + correct KP supervision)

In [ ]:
def ids_to_tokens(ids, idx2tok):
    return [idx2tok.get(int(i), f"<unk:{int(i)}>") for i in ids]

row = main_df.iloc[1]

userid = str(row["userid"])
task   = str(row["task"]).strip()
task_id = int(row["task_id"])

# ---------- action sequence ----------
actions = parse_serialized_sequence(row["interactions"])
if len(actions) == 0:
    raise ValueError("Empty interactions in this row.")
action_ids = [actions2idx[a] for a in actions]
T = len(action_ids)
action_labels = torch.tensor(action_ids, device=device).unsqueeze(0)  # (1,T)

# ---------- node sequence (history) ----------
node_seq = parse_serialized_sequence(row["text_nodes"])
if len(node_seq) != T:
    raise ValueError(f"Mismatch: len(text_nodes)={len(node_seq)} vs len(interactions)={T} in row_id={row.get('row_id','?')}")

node_seq = [normalize_id(n) for n in node_seq]
node_label_ids = [node2idx.get(n, None) for n in node_seq]
if any(x is None for x in node_label_ids):
    missing = [n for n, x in zip(node_seq, node_label_ids) if x is None][:10]
    raise ValueError(f"Some node ids not found in node2idx (first 10): {missing}")

node_labels = torch.tensor(node_label_ids, device=device).unsqueeze(0)  # (1,T)

# ---------- e_seq embeddings from node ids ----------
e_seq_np = np.stack([emb_resolver.get(n) for n in node_seq], axis=0)     # (T,768)
e_seq = torch.tensor(e_seq_np, device=device).unsqueeze(0)              # (1,T,768)

# ---------- query embedding ----------
query_text_id = normalize_id(row["query_text_id"])
q_emb_np = emb_resolver.get(query_text_id)
q_text_emb = torch.tensor(q_emb_np, device=device).unsqueeze(0)         # (1,768)

# ---------- user/task labels ----------
user_labels = torch.tensor([user2idx[userid]], device=device)           # (1,)
task_labels = torch.tensor([task_id], device=device)                    # (1,)

# ---------- KP targets: use target_output_id for generation tasks ----------
kp_id = get_kp_supervision_id(row)
if kp_id is None:
    kp_targets = torch.zeros(1, len(kp2idx), device=device)
    kp_mask = torch.zeros(1, device=device)  # no kp loss
    gold_phrases = []
else:
    kp_y, gold_phrases = build_kp_target_from_id(kp_id, id2phrases, kp2idx, device=device)
    kp_targets = kp_y.unsqueeze(0)
    kp_mask = torch.ones(1, device=device)

# ---------- forward + one optimizer step ----------
model.train()

# build one-hot for modulation
B, T = action_labels.shape
action_onehot = F.one_hot(action_labels, num_classes=len(actions2idx)).float()

out = model(
    e_seq=e_seq,
    action_onehot=action_onehot,   # <-- REQUIRED
    action_labels=action_labels,
    node_labels=node_labels,
    task_labels=task_labels,
    user_labels=user_labels,
    q_text_emb=q_text_emb,
    kp_targets=kp_targets,
    kp_loss_mask=kp_mask
)

optim.zero_grad()
out["loss_total"].backward()
optim.step()


# ---------- decode predicted keyphrases ----------
model.eval()
pred_phrases = model.decoder.decode(out["kp_logits"], idx2kp=idx2kp, topk=15, threshold=0.3)

# ---------- print readable outputs ----------
true_user_id = int(user_labels.item())
pred_user_id = int(out["pred_user"].item())

pred_task_id = int(out["pred_task"].item()) if "pred_task" in out else None

print("\n---- SAMPLE 1 ROW ----")
print("row_id:", row.get("row_id", None))
print("userid:", userid, "| true_user:", idx2user[true_user_id], "| pred_user:", idx2user.get(pred_user_id, f"<unk_user:{pred_user_id}>"))
print("task:", task, "| true_task:", idx2task[task_id], "| pred_task:", idx2task.get(pred_task_id, f"<unk_task:{pred_task_id}>"))

print("\nACTIONS:")
print("  true_actions (first 10):", actions[:10])
print("  pred_actions (first 10):", ids_to_tokens(out["pred_actions"][0, :10].tolist(), idx2action))

print("\nNODES:")
print("  true_nodes (first 10):", node_seq[:10])
print("  pred_nodes (first 10):", ids_to_tokens(out["pred_nodes"][0, :10].tolist(), idx2node))

print("\nKP SUPERVISION:")
print("  kp_id used for gold:", kp_id)
print("  gold_phrases (top 15):", gold_phrases)

print("\nKP PRED:")
for ph, pr in pred_phrases[0]:
    print(f"  {ph} ({pr:.4f})")

print("\nLOSSES:")
print("  loss_action =", float(out.get("loss_action", torch.tensor(0.0)).item()))
print("  loss_node   =", float(out.get("loss_node", torch.tensor(0.0)).item()))
print("  loss_task   =", float(out.get("loss_task", torch.tensor(0.0)).item()) if "loss_task" in out else None)
print("  loss_user   =", float(out.get("loss_user", torch.tensor(0.0)).item()) if "loss_user" in out else None)
print("  loss_kp     =", float(out.get("loss_kp", torch.tensor(0.0)).item()) if "loss_kp" in out else None)
print("  loss_total  =", float(out["loss_total"].item()))
print("  kp_mask     =", float(kp_mask.item()))

rain/test split (70–30) + basic setup

In [ ]:
from tqdm.auto import tqdm
import os
import math

# ----------------------------
# 70-30 split
# ----------------------------
rng = np.random.default_rng(42)
idx = np.arange(len(main_df))
rng.shuffle(idx)

# Ensure split column is normalized
main_df["split"] = main_df["split"].str.strip().str.lower()

# Create train and test DataFrames
train_df = main_df[main_df["split"] == "train"].reset_index(drop=True)
test_df  = main_df[main_df["split"] == "test"].reset_index(drop=True)

print("Train size:", len(train_df), "| Test size:", len(test_df))


# print("Train size:", len(train_df), "| Test size:", len(test_df))

# ----------------------------
# Checkpoint folder
# ----------------------------
CKPT_DIR = r"/teamspace/studios/this_studio/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
print("Checkpoint dir:", CKPT_DIR)

Batch builder (row → tensors)

In [ ]:
MAX_T = 100

def build_row_tensors(row, device):
    """
    Same as before, BUT:
    - If |sequence| > 100, keep ONLY last 100 tokens
    """

    # --------- required fields ----------
    userid = str(row["userid"]).strip()
    if userid not in user2idx:
        raise ValueError(f"userid not in user2idx: {userid}")

    task = str(row["task"]).strip()
    if task not in task2idx:
        raise ValueError(f"task not in task2idx: {task}")

    task_id = int(row["task_id"])
    query_text_id = normalize_id(row["query_text_id"])

    # --------- parse sequences ----------
    actions = parse_serialized_sequence(row["interactions"])
    node_seq = parse_serialized_sequence(row["text_nodes"])

    if len(actions) == 0:
        raise ValueError("empty interactions")
    if len(node_seq) == 0:
        raise ValueError("empty text_nodes")
    if len(actions) != len(node_seq):
        raise ValueError(f"length mismatch: interactions={len(actions)} text_nodes={len(node_seq)}")

    # ===============================
    # KEEP ONLY LAST 100 STEPS
    # ===============================
    if len(actions) > MAX_T:
        actions = actions[-MAX_T:]
        node_seq = node_seq[-MAX_T:]

    T = len(actions)

    # --------- action labels ----------
    action_ids = []
    for a in actions:
        if a not in actions2idx:
            raise ValueError(f"unknown action token: {a}")
        action_ids.append(actions2idx[a])

    action_labels = torch.tensor(action_ids, device=device).unsqueeze(0)  # (1,T)

    # --------- node labels ----------
    node_seq = [normalize_id(n) for n in node_seq]
    node_ids = []
    missing_nodes = []

    for n in node_seq:
        j = node2idx.get(n, None)
        if j is None:
            missing_nodes.append(n)
        else:
            node_ids.append(j)

    if missing_nodes:
        raise ValueError(f"missing node ids in node2idx: {missing_nodes[:10]}")

    node_labels = torch.tensor(node_ids, device=device).unsqueeze(0)  # (1,T)

    # --------- e_seq embeddings ----------
    e_list = []
    for n in node_seq:
        emb = emb_resolver.get(n)
        if emb is None:
            raise ValueError(f"missing embedding for node: {n}")
        e_list.append(emb)

    e_seq_np = np.stack(e_list, axis=0)              # (T,768)
    e_seq = torch.tensor(e_seq_np, device=device).unsqueeze(0)  # (1,T,768)

    # --------- query embedding ----------
    q_np = emb_resolver.get(query_text_id)
    if q_np is None:
        raise ValueError(f"missing embedding for query_text_id: {query_text_id}")

    q_text_emb = torch.tensor(q_np, device=device).unsqueeze(0)   # (1,768)

    # --------- user/task labels ----------
    user_labels = torch.tensor([user2idx[userid]], device=device)  # (1,)
    task_labels = torch.tensor([task_id], device=device)           # (1,)

    # --------- KP targets ----------
    kp_id = get_kp_supervision_id(row)  # None if not gen task
    if kp_id is None:
        kp_targets = None
        kp_mask = torch.tensor([0.0], device=device)
        gold_phrases = []
    else:
        if kp_id not in id2phrases:
            kp_targets = None
            kp_mask = torch.tensor([0.0], device=device)
            gold_phrases = []
        else:
            y, gold_phrases = build_kp_target_from_id(
                kp_id, id2phrases, kp2idx, device=device
            )
            kp_targets = y.unsqueeze(0)      # (1,|kp|)
            kp_mask = torch.tensor([1.0], device=device)

    meta = {
        "userid": userid,
        "task": task,
        "query_text_id": query_text_id,
        "kp_id": kp_id,
        "actions": actions,
        "node_seq": node_seq,
        "gold_kp_phrases": gold_phrases
    }

    return {
        "e_seq": e_seq,
        "action_labels": action_labels,
        "node_labels": node_labels,
        "task_labels": task_labels,
        "user_labels": user_labels,
        "q_text_emb": q_text_emb,
        "kp_targets": kp_targets,
        "kp_loss_mask": kp_mask,
        "meta": meta
    }


def is_finite_tensor(x: torch.Tensor) -> bool:
    return torch.isfinite(x).all().item()

def grads_are_finite(model: nn.Module) -> bool:
    for p in model.parameters():
        if p.grad is None:
            continue
        if not torch.isfinite(p.grad).all():
            return False
    return True

def grad_global_norm(model: nn.Module) -> float:
    total = 0.0
    for p in model.parameters():
        if p.grad is None:
            continue
        total += p.grad.detach().float().pow(2).sum().item()
    return float(math.sqrt(total))


Training loop (5 epochs) with tqdm, skipping bad rows, clipping, checkpoints

In [ ]:
EPOCHS = 4
CLIP_NORM = 0.75

# For running loss stats
LOSS_KEYS = ["loss_total", "loss_action", "loss_node", "loss_task", "loss_user", "loss_kp"]

# ===============================
# RESUME FROM CHECKPOINT (IF ANY)
# ===============================
START_EPOCH = 1

if os.path.isdir(CKPT_DIR):
    ckpts = [
        f for f in os.listdir(CKPT_DIR)
        if f.startswith("unihis_epoch_") and f.endswith(".pt")
    ]
    if len(ckpts) > 0:
        # pick latest epoch
        ckpts = sorted(
            ckpts,
            key=lambda x: int(x.replace("unihis_epoch_", "").replace(".pt", ""))
        )
        last_ckpt = ckpts[-1]
        ckpt_path = os.path.join(CKPT_DIR, last_ckpt)

        print(f"Resuming from checkpoint: {ckpt_path}")
        ckpt = torch.load(ckpt_path, map_location=device)

        model.load_state_dict(ckpt["model_state_dict"], strict=True)
        optim.load_state_dict(ckpt["optim_state_dict"])

        START_EPOCH = ckpt["epoch"] + 1

        print(f"Resuming training from epoch {START_EPOCH}")
    else:
        print("No checkpoint found. Training from scratch.")
else:
    print("Checkpoint directory does not exist. Training from scratch.")


for epoch in range(1, EPOCHS + 1):
    model.train()

    running = {k: 0.0 for k in LOSS_KEYS}
    steps_ok = 0
    steps_skipped = 0
    rows_bad = 0

    pbar = tqdm(range(len(train_df)), desc=f"Epoch {epoch}/{EPOCHS} [train]", dynamic_ncols=True)

    for i in pbar:
        row = train_df.iloc[i]

        # ---------- build tensors ----------
        try:
            batch = build_row_tensors(row, device=device)
        except Exception as e:
            rows_bad += 1
            steps_skipped += 1
            continue

        # ---------- forward ----------
        # Build one-hot for training
        action_ids = batch["action_labels"]           # (1,T)
        action_onehot = F.one_hot(
            action_ids,
            num_classes=len(actions2idx)
        ).float().to(device)

        out = model(
            e_seq=batch["e_seq"],
            action_onehot=action_onehot,   # <-- REQUIRED
            action_labels=batch["action_labels"],
            node_labels=batch["node_labels"],
            task_labels=batch["task_labels"],
            user_labels=batch["user_labels"],
            q_text_emb=batch["q_text_emb"],
            kp_targets=batch["kp_targets"],
            kp_loss_mask=batch["kp_loss_mask"],
        )


        loss = out["loss_total"]

        # Skip if loss is NaN/Inf
        if not is_finite_tensor(loss):
            steps_skipped += 1
            optim.zero_grad(set_to_none=True)
            continue

        # ---------- backward ----------
        optim.zero_grad(set_to_none=True)
        loss.backward()

        # Sanity: skip if any grad NaN/Inf
        if not grads_are_finite(model):
            steps_skipped += 1
            optim.zero_grad(set_to_none=True)
            continue

        # ---------- clip + step ----------
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
        optim.step()

        # ---------- update running stats ----------
        steps_ok += 1

        running["loss_total"] += float(out["loss_total"].item())
        running["loss_action"] += float(out.get("loss_action", torch.tensor(0.0, device=device)).item())
        running["loss_node"]   += float(out.get("loss_node", torch.tensor(0.0, device=device)).item())
        running["loss_task"]   += float(out.get("loss_task", torch.tensor(0.0, device=device)).item())
        running["loss_user"]   += float(out.get("loss_user", torch.tensor(0.0, device=device)).item())
        running["loss_kp"]     += float(out.get("loss_kp", torch.tensor(0.0, device=device)).item())

        # ---------- progress bar display ----------
        denom = max(1, steps_ok)
        pbar.set_postfix({
            "L": f"{running['loss_total']/denom:.3f}",
            "La": f"{running['loss_action']/denom:.3f}",
            "Ln": f"{running['loss_node']/denom:.3f}",
            "Lt": f"{running['loss_task']/denom:.3f}",
            "Lu": f"{running['loss_user']/denom:.3f}",
            "Lkp": f"{running['loss_kp']/denom:.3f}",
            "skip": steps_skipped
        })

    # ---------- end of epoch: print summary ----------
    denom = max(1, steps_ok)
    print(f"\nEpoch {epoch} TRAIN SUMMARY")
    print(f"  steps_ok={steps_ok}  steps_skipped={steps_skipped}  rows_bad={rows_bad}")
    print(f"  loss_total={running['loss_total']/denom:.4f}")
    print(f"    loss_action={running['loss_action']/denom:.4f}")
    print(f"    loss_node  ={running['loss_node']/denom:.4f}")
    print(f"    loss_task  ={running['loss_task']/denom:.4f}")
    print(f"    loss_user  ={running['loss_user']/denom:.4f}")
    print(f"    loss_kp    ={running['loss_kp']/denom:.4f}")

    # ---------- save checkpoint ----------
    ckpt_path = os.path.join(CKPT_DIR, f"unihis_epoch_{epoch}.pt")
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optim_state_dict": optim.state_dict(),
        "vocab_sizes": {
            "users": len(user2idx),
            "tasks": n_tasks,
            "actions": len(actions2idx),
            "nodes": len(node2idx),
            "keyphrases": len(kp2idx),
        },
        "loss_train_avg": {k: running[k]/denom for k in running},
    }, ckpt_path)
    print("Saved checkpoint:", ckpt_path)

Inference on test split with (task-specific printing)

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from collections import defaultdict

# ============================
# Load latest checkpoint
# ============================
ckpt_files = sorted(
    [f for f in os.listdir(CKPT_DIR) if f.startswith("unihis_epoch_") and f.endswith(".pt")],
    key=lambda x: int(x.split("_")[-1].split(".")[0])
)
assert len(ckpt_files) > 0, "No checkpoints found."

latest_ckpt = os.path.join(CKPT_DIR, ckpt_files[-1])
ckpt = torch.load(latest_ckpt, map_location=device)

model.load_state_dict(ckpt["model_state_dict"])
model.to(device)
model.eval()

print("Loaded checkpoint:", latest_ckpt)


@torch.no_grad()
def run_single_row(row):
    userid  = str(row["userid"])
    task    = str(row["task"]).strip()
    task_id = int(row["task_id"])

    # ---------- actions ----------
    actions = parse_serialized_sequence(row["interactions"])
    if len(actions) == 0:
        return None

    action_ids = [actions2idx[a] for a in actions]
    action_labels = torch.tensor(action_ids, device=device).unsqueeze(0)

    # ---------- nodes ----------
    node_seq = parse_serialized_sequence(row["text_nodes"])
    if len(node_seq) != len(actions):
        return None

    node_seq = [normalize_id(n) for n in node_seq]
    node_label_ids = [node2idx.get(n, None) for n in node_seq]
    if any(x is None for x in node_label_ids):
        return None

    node_labels = torch.tensor(node_label_ids, device=device).unsqueeze(0)

    # ---------- e_seq ----------
    e_seq_np = np.stack([emb_resolver.get(n) for n in node_seq], axis=0)
    e_seq = torch.tensor(e_seq_np, device=device).unsqueeze(0)

    # ---------- query ----------
    query_text_id = normalize_id(row["query_text_id"])
    q_emb_np = emb_resolver.get(query_text_id)
    q_text_emb = torch.tensor(q_emb_np, device=device).unsqueeze(0)

    # ---------- labels ----------
    user_labels = torch.tensor([user2idx[userid]], device=device)
    task_labels = torch.tensor([task_id], device=device)

    # ---------- gold keyphrases ----------
    kp_id = get_kp_supervision_id(row)
    if kp_id is None:
        gold_kps = []
    else:
        _, gold_kps = build_kp_target_from_id(
            kp_id, id2phrases, kp2idx, device=device
        )

    # ---------- forward ----------
    action_onehot = F.one_hot(
        action_labels, num_classes=len(actions2idx)
    ).float()

    out = model(
        e_seq=e_seq,
        action_onehot=action_onehot,
        action_labels=None,
        node_labels=None,
        task_labels=task_labels,
        user_labels=user_labels,
        q_text_emb=q_text_emb,
        kp_targets=None,
        kp_loss_mask=None
    )

    pred_task_id = int(out["pred_task"].item())
    pred_task = idx2task[pred_task_id]

    result = {
        "true_task": task,
        "pred_task": pred_task
    }

    # ======================================================
    # NEXT USER PREDICTION — TOP-15 WITH PROBABILITIES
    # ======================================================
    if task == "next_user_prediction":
        true_target = idx2user[user_labels.item()]

        P_user = out["P_user"]                      # (1, n_users)
        topk_prob, topk_idx = P_user.topk(15, dim=-1)

        top15 = [
            (idx2user[i.item()], p.item())
            for i, p in zip(topk_idx[0], topk_prob[0])
        ]

        result.update({
            "true_target_id": true_target,
            "predicted_target_id": top15[0][0],
            "top15_predicted_targets": top15
        })

    # ======================================================
    # NEXT ITEM PREDICTION — TOP-15 WITH PROBABILITIES
    # ======================================================
    elif task == "next_item_prediction":
        true_target = idx2node[node_labels[0, -1].item()]

        node_logits_last = out["node_logits"][:, -1, :]   # (1, n_nodes)
        P_node = F.softmax(node_logits_last, dim=-1)

        topk_prob, topk_idx = P_node.topk(15, dim=-1)

        top15 = [
            (idx2node[i.item()], p.item())
            for i, p in zip(topk_idx[0], topk_prob[0])
        ]

        result.update({
            "true_target_id": true_target,
            "predicted_target_id": top15[0][0],
            "top15_predicted_targets": top15
        })

    # ======================================================
    # GENERATION TASKS
    # ======================================================
    elif task in {"personalized_summarization", "dialogue_generation"}:
        pred_kps = model.decoder.decode(
            out["kp_logits"], idx2kp=idx2kp, topk=15, threshold=0.0005
        )[0]

        result.update({
            "gold_keyphrases": gold_kps,
            "predicted_keyphrases": [f"{p}({prob:.2f})" for p, prob in pred_kps]
        })

    return result



# ============================
# Collect ONE sample per task
# ============================
seen_tasks = set()
results = []

for _, row in test_df.iterrows():
    task = str(row["task"]).strip()
    if task in seen_tasks:
        continue

    out = run_single_row(row)
    if out is None:
        continue

    results.append(out)
    seen_tasks.add(task)

    if len(seen_tasks) == len(test_df["task"].unique()):
        break


# ============================
# Print results
# ============================
print("\n===== ONE SAMPLE PER TASK (TEST SET) =====")
for r in results:
    print("\n----------------------------------------")
    print("True task :", r["true_task"])
    print("Pred task :", r["pred_task"])

    if "true_target_id" in r:
        print("True target id :", r["true_target_id"])
        print("Predicted target id :", r["predicted_target_id"])

        if "top15_predicted_targets" in r:
            print("Top-15 predicted targets (with probabilities):")
            for tok, prob in r["top15_predicted_targets"]:
                print(f"  {tok} ({prob:.4f})")

    if "gold_keyphrases" in r:
        print("Gold keyphrases:")
        print(" ", r["gold_keyphrases"])
        print("Predicted keyphrases (top-15):")
        for x in r["predicted_keyphrases"]:
            print(" ", x)



In [ ]:
import math
from collections import defaultdict

def dcg_at_k(rel, k):
    return sum(
        rel[i] / math.log2(i + 2) for i in range(min(k, len(rel)))
    )

def ndcg_at_k(rank_list, gt_set, k):
    rel = [1 if x in gt_set else 0 for x in rank_list[:k]]
    dcg = dcg_at_k(rel, k)
    idcg = dcg_at_k(sorted(rel, reverse=True), k)
    return dcg / idcg if idcg > 0 else 0.0

def mrr(rank_list, gt_set):
    for i, x in enumerate(rank_list):
        if x in gt_set:
            return 1.0 / (i + 1)
    return 0.0

def hit_at_k(rank_list, gt_set, k):
    return int(any(x in gt_set for x in rank_list[:k]))

def precision_recall(pred_set, gold_set):
    if len(pred_set) == 0:
        return 0.0, 0.0
    tp = len(pred_set & gold_set)
    prec = tp / len(pred_set)
    rec = tp / len(gold_set) if len(gold_set) > 0 else 0.0
    return prec, rec


MAX_PER_TASK = 30

stats = defaultdict(lambda: defaultdict(list))
task_counts = defaultdict(int)

for _, row in test_df.iterrows():
    task = str(row["task"]).strip()
    if task_counts[task] >= MAX_PER_TASK:
        continue

    out = run_single_row(row)
    if out is None:
        continue

    task_counts[task] += 1

    # ======================================================
    # NEXT USER / NEXT ITEM — RANKING METRICS
    # ======================================================
    if task in {"next_user_prediction", "next_item_prediction"}:
        true_id = out["true_target_id"]
        ranked = [x for x, _ in out["top15_predicted_targets"]]

        gt = {true_id}

        stats[task]["ndcg5"].append(ndcg_at_k(ranked, gt, 5))
        stats[task]["ndcg10"].append(ndcg_at_k(ranked, gt, 10))
        stats[task]["mrr"].append(mrr(ranked, gt))
        stats[task]["hit5"].append(hit_at_k(ranked, gt, 5))

    # ======================================================
    # GENERATION TASKS — PRECISION / RECALL
    # ======================================================
    elif task in {"personalized_summarization", "dialogue_generation"}:
        gold = set(out["gold_keyphrases"])
        pred = set(
            p.split("(")[0] for p in out["predicted_keyphrases"]
        )

        prec, rec = precision_recall(pred, gold)
        stats[task]["precision"].append(prec)
        stats[task]["recall"].append(rec)

    if all(task_counts[t] >= MAX_PER_TASK for t in test_df["task"].unique()):
        break


print("\n========== EVALUATION SUMMARY (30 rows per task) ==========")

for task, m in stats.items():
    print(f"\nTask: {task}")

    if "ndcg5" in m:
        print(f"  nDCG@5 : {sum(m['ndcg5']) / len(m['ndcg5']):.4f}")
        print(f"  nDCG@10: {sum(m['ndcg10']) / len(m['ndcg10']):.4f}")
        print(f"  MRR    : {sum(m['mrr']) / len(m['mrr']):.4f}")
        print(f"  Hit@5  : {sum(m['hit5']) / len(m['hit5']):.4f}")

    if "precision" in m:
        print(f"  Precision: {sum(m['precision']) / len(m['precision']):.4f}")
        print(f"  Recall   : {sum(m['recall']) / len(m['recall']):.4f}")
